[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/rag-chatbot.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/rag-chatbot.ipynb)

# Contrução de Chatbot de IA com LangChain, API do Gemini e API do Pinecone

Este protótipo implementa um chatbot de IA utilizando o paradigma RAG (Retrieval Augmented Generation), permitindo que o modelo acesse informações de uma base de conhecimento externa durante o processo de geração de respostas. A arquitetura emprega LangChain para orquestração das etapas de recuperação e geração, e um banco vetorial destinado ao armazenamento e à consulta eficiente de embeddings.

O dataset utilizado contém documentos relacionados a temas atuais de inteligência artificial, garantindo que o chatbot possa responder de maneira contextualizada e alinhada ao conteúdo disponível. A camada de LLM opera por meio da API do Gemini, enquanto o armazenamento e a recuperação dos vetores são realizados utilizando a API do Pinecone.

### Antes de Iniciar

É necessário uma [chave de API do Gemini](https://aistudio.google.com/app/api-keys) and [chave de API do Pinecone](https://app.pinecone.io).

### Prerequisitos

Antes de iniciar a construção do chatbot, é necessário instalar as bibliotecas Python utilizadas no protótipo. A seguir, apresentamos uma breve descrição da função de cada uma:

- **langchain**: Biblioteca para orquestração de pipelines de IA generativa. Será utilizada para integrar os componentes de recuperação e geração do chatbot.

- **google-generativeai**: Cliente oficial para acesso à API do Gemini. Responsável pela camada de LLM utilizada para gerar respostas.

- **pinecone-client**: Cliente oficial da Pinecone. Permite a conexão com o serviço de banco vetorial onde serão armazenados e consultados os embeddings.

- **bs4**: Biblioteca para webscraping para adquirir o texto da lei a ser usada no RAG.

Você pode instalar essas bibliotecas com o pip:

In [52]:
!pip install -qU \
    langchain==0.3.23 \
    langchain-community==0.3.21 \
    langchain-pinecone==0.2.5 \
    langchain-google-genai \
    datasets==3.5.0 \
    bs4

### Construindo um Chatbot (sem RAG)

Vamos utilizar amplamente a biblioteca LangChain para integrar os diferentes componentes necessários ao funcionamento do chatbot. Para iniciar, construiremos um chatbot simples sem RAG, utilizando apenas a camada de LLM. Isso é feito inicializando um objeto de modelo conversacional compatível com o Gemini, que atuará como gerador de respostas.

In [119]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or getpass(
    "Enter your Gemini API key: "
)

chat = ChatGoogleGenerativeAI(
    google_api_key=os.environ["GOOGLE_API_KEY"],
    model='gemini-2.5-flash'
)

Enter your Gemini API key: ··········


Chats com o modelo Gemini 2.5 Flash normalmente são estruturados (em texto simples) assim:

```
System: Você é um assistente útil.

User: Oi IA, como você está hoje?

Model: Estou ótimo, obrigado. Em que posso ajudar?

User: Gostaria de entender teoria das cordas.

Model:
```
A linha final "Model:" sem resposta é o que leva o modelo a continuar a conversa.

Na API oficial do Google Generative AI, isso seria enviado ao modelo neste formato:

```python
[
    {"role": "system", "parts": "Você é um agente de IA especializado em gerenciamento de projetos. Você é especialista em metodologias ágeis, escopo creep, PRINCE 2, PMBOK 7 e gerenciamento de riscos."},
    {"role": "user",  "parts": "Oi IA, como você está hoje?"},
    {"role": "model", "parts": "Estou ótimo, obrigado. Em que posso ajudar?"},
    {"role": "user",  "parts": "O que é escopo?"}
]
```

No LangChain, existe um formato um pouco diferente. Usamos três objetos de mensagem, assim:

In [10]:
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

messages = [
    SystemMessage(content="Você é um agente de IA especializado em gerenciamento de projetos. Você é especialista em metodologias ágeis, escopo creep, PRINCE 2, PMBOK 7 e gerenciamento de riscos."),
    HumanMessage(content="Olá IA, como você está?"),
    AIMessage(content="Estou bem, obrigado. Como posso te ajudar?"),
    HumanMessage(content="O que é escopo?")
]

O formato é muito semelhante — apenas trocamos o papel de "user" por HumanMessage, e o papel de "model" por AIMessage.

Geramos a próxima resposta da IA passando essas mensagens para o objeto ChatGoogleGenerativeAI:

In [11]:
res = chat.invoke(messages)
res

AIMessage(content='Em gerenciamento de projetos, o **escopo** refere-se ao trabalho que precisa ser realizado para entregar um produto, serviço ou resultado com as características e funções especificadas. É fundamentalmente a definição dos limites do projeto – o que está incluído e, igualmente importante, o que *não* está incluído.\n\nPodemos dividi-lo em duas categorias principais:\n\n1.  **Escopo do Produto (Product Scope):**\n    *   Descreve as características, funções e requisitos do produto, serviço ou resultado que o projeto irá entregar.\n    *   Responde à pergunta: "O que o cliente receberá no final?"\n    *   Exemplo: Para um software, o escopo do produto incluiria funcionalidades como "login de usuário", "carrinho de compras", "gerenciamento de estoque", etc.\n\n2.  **Escopo do Projeto (Project Scope):**\n    *   Descreve o trabalho que precisa ser realizado para entregar o escopo do produto.\n    *   Responde à pergunta: "Qual trabalho faremos para criar o produto, serviço

Na resposta, recebemos outro objeto AIMessage. Podemos exibi-lo de forma mais clara assim:

In [12]:
print(res.content)

Em gerenciamento de projetos, o **escopo** refere-se ao trabalho que precisa ser realizado para entregar um produto, serviço ou resultado com as características e funções especificadas. É fundamentalmente a definição dos limites do projeto – o que está incluído e, igualmente importante, o que *não* está incluído.

Podemos dividi-lo em duas categorias principais:

1.  **Escopo do Produto (Product Scope):**
    *   Descreve as características, funções e requisitos do produto, serviço ou resultado que o projeto irá entregar.
    *   Responde à pergunta: "O que o cliente receberá no final?"
    *   Exemplo: Para um software, o escopo do produto incluiria funcionalidades como "login de usuário", "carrinho de compras", "gerenciamento de estoque", etc.

2.  **Escopo do Projeto (Project Scope):**
    *   Descreve o trabalho que precisa ser realizado para entregar o escopo do produto.
    *   Responde à pergunta: "Qual trabalho faremos para criar o produto, serviço ou resultado?"
    *   Inclui

Como res é apenas outro objeto AIMessage, podemos adicioná-lo à lista messages, incluir um novo HumanMessage e gerar a próxima resposta na conversa:

In [44]:
messages.append(res)

prompt = HumanMessage(
    content="Porque definir o escopo de um projeto pode ser uma tarefa tão complexa?"
)

messages.append(prompt)

res = chat.invoke(messages)

print(res.content)

Você acertou em cheio! Definir o escopo de um projeto é, de fato, uma das tarefas mais complexas e críticas no gerenciamento de projetos. A complexidade surge de uma série de fatores interligados, que podem ser agrupados em algumas categorias principais:

1.  **Natureza Humana e Stakeholders:**
    *   **Expectativas Divergentes:** Diferentes stakeholders (clientes, usuários finais, patrocinadores, equipe interna) podem ter ideias muito distintas sobre o que o projeto deve entregar. Conciliar essas visões e chegar a um consenso é um desafio enorme.
    *   **Dificuldade em Articular Necessidades:** Muitas vezes, os próprios clientes não sabem exatamente o que querem ou precisam, ou têm dificuldade em expressar suas necessidades de forma clara e detalhada. Eles podem descrever um problema, mas não a solução ideal ou os requisitos específicos.
    *   **Interesses Conflitantes:** Stakeholders podem ter agendas ou prioridades que se chocam, levando a debates e dificuldades na tomada de de

### Lidando com Alucinações

Temos nosso chatbot, mas como mencionado — o conhecimento dos LLMs pode ser limitado.
A razão para isso é que os LLMs aprendem tudo o que sabem durante o treinamento.
Um LLM basicamente comprime o “mundo” visto nos dados de treinamento dentro dos parâmetros internos do modelo.
Chamamos esse conhecimento de conhecimento paramétrico do modelo.

Por padrão, LLMs não têm acesso ao mundo externo.

O resultado disso fica muito evidente quando pedimos a eles informações mais específicas.

In [137]:
messages = []

prompt = HumanMessage(
    content="Quais entidades esportivas podem receber recursos das loterias e que tipo de prestação de contas elas devem apresentar para garantir a integridade no uso desses recursos?"
)
messages.append(prompt)

res = chat.invoke(messages)

In [138]:
print(res.content)

No Brasil, a distribuição dos recursos provenientes das loterias federais para o esporte é regulamentada principalmente pela **Lei nº 9.615/1998 (Lei Pelé)** e, mais recentemente, pela **Lei nº 13.756/2018**, que detalhou as destinações.

### Entidades Esportivas que Podem Receber Recursos das Loterias:

As principais entidades esportivas que recebem recursos das loterias são:

1.  **Comitê Olímpico do Brasil (COB):** Recebe um percentual dos recursos para o desenvolvimento do esporte olímpico, preparação de atletas, participação em competições internacionais, etc.
2.  **Comitê Paralímpico Brasileiro (CPB):** Da mesma forma, recebe um percentual para o fomento e desenvolvimento do esporte paralímpico, incluindo a preparação de atletas e a participação em eventos.
3.  **Confederações Nacionais de Administração do Desporto:** São as entidades máximas de cada modalidade esportiva (ex: Confederação Brasileira de Futebol - CBF, Confederação Brasileira de Voleibol - CBV, etc.). Elas recebem 

Nosso chatbot não consegue mais nos ajudar, ele não tem a informação necessária para responder à pergunta. Ficou muito claro pela resposta que o LLM não sabe essa informação, mas às vezes um LLM pode responder como se soubesse, e isso pode ser muito difícil de detectar.

Existe outra forma de fornecer conhecimento a um LLM.
Chamamos isso de conhecimento de fonte (source knowledge), que se refere a qualquer informação inserida no LLM por meio do prompt. Nós podemos testar isso com a pergunta anterior, adicionamos um trecho da Lei Nº 13.756, de 12 de Dezembro De 2018.

In [139]:
source_knowledge = (
    "Dispõe sobre o Fundo Nacional de Segurança Pública (FNSP), sobre a "
    "destinação do produto da arrecadação das loterias e sobre a promoção  "
    "comercial e a modalidade lotérica denominada apostas de quota fixa; altera as "
    "Leis n º 8.212, de 24 de julho de 1991, 9.615, de 24 março de 1998, 10.891, de  "
    "9 de julho de 2004, 11.473, de 10 de maio de 2007, e 13.675, de 11 de junho de 2018; "
)

Podemos inserir esse conhecimento adicional no nosso prompt junto com instruções dizendo ao LLM como queremos que ele use essas informações em conjunto com a nossa pergunta original.

In [147]:
query = "Quais entidades esportivas podem receber recursos das loterias e que tipo de prestação de contas elas devem apresentar para garantir a integridade no uso desses recursos?"

augmented_prompt = f"""Usando os contextos, responda a pergunta.

Contextos:
{source_knowledge}

Pergunta: {query}"""

Agora alimentamos isso no nosso chatbot da mesma forma que fizemos antes.

In [148]:
prompt = HumanMessage(
    content=augmented_prompt
)

messages.append(prompt)

res = chat.invoke(messages)

In [149]:
print(res.content)

Com base exclusivamente no contexto fornecido, as informações são limitadas para responder à pergunta.

1.  **Quais entidades esportivas podem receber recursos das loterias?**
    O contexto menciona que o diploma legal altera, entre outras, a **Lei nº 9.615, de 24 de março de 1998**, que é a Lei Geral do Esporte (Lei Pelé). Essa menção **sugere** que há uma relação entre a destinação dos recursos das loterias e o setor esportivo, e que as regras para tal destinação ou as entidades beneficiárias podem ser afetadas por essa alteração. No entanto, o texto **não especifica quais entidades esportivas em particular** (confederações, federações, clubes, atletas, etc.) podem receber esses recursos.

2.  **Que tipo de prestação de contas elas devem apresentar para garantir a integridade no uso desses recursos?**
    O contexto disponibilizado **não contém informações** sobre o tipo de prestação de contas que as entidades (esportivas ou de qualquer outro tipo) devem apresentar para garantir a i

Agora o LLM responde com base no contexto que adicionamos. O resultado ainda não está muito legal, pois o contexto adicionando foi muito pequeno.

### Importando os Dados

Vamos usar o Beautiful Soup para fazer scraping da página da Lei Nº 13.756, de 12 de Dezembro De 2018. Depois, iremos usar esse texto com uma base de conhecimento do chatbot

In [53]:
LEI_13756_URL = "https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13756.htm"

In [62]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

bs4_strainer = bs4.SoupStrainer()
loader = WebBaseLoader(
    web_paths=(LEI_13756_URL,),
    bs_kwargs={"parse_only": bs4_strainer},
        requests_kwargs={
        "headers": {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/129.0 Safari/537.36"
            )
        },
        "timeout": 20,
    },
)

In [74]:
docs = loader.load()
print(f"Total de caracteres: {len(docs[0].page_content)}")

Total de caracteres: 141015


In [95]:
import re

docs[0].page_content = re.sub(r"\n{3,}", "\n", docs[0].page_content)

### Tarefa 4: Construindo a Base de Conhecimento

Agora que já temos um dataset que servirá como base de conhecimento do chatbot, o próximo passo é transformá-lo em um formato utilizável pelo sistema. Para isso, utilizamos um modelo de embeddings combinado com um banco vetorial, permitindo a indexação e a recuperação eficiente das informações durante o processo de RAG.

Iniciamos essa etapa configurando o cliente do Pinecone, responsável pelo armazenamento vetorial dos embeddings. Para isso, é necessária uma chave de API do Pinecone, disponível gratuitamente (com acesso limitado).

In [76]:
from pinecone import Pinecone

api_key = os.getenv("PINECONE_API_KEY") or getpass(
    "Enter your Pinecone API key: "
)

pc = Pinecone(api_key=api_key)

Enter your Pinecone API key: ··········


Isso nos permite definir o provedor de nuvem e a região onde queremos implantar o índice.
Você pode encontrar a lista de todos os provedores e regiões disponíveis na [documentação](https://docs.pinecone.io/troubleshooting/available-cloud-regions).

In [154]:
from pinecone import ServerlessSpec, CloudProvider, AwsRegion

index_name = "gemini-pf-rag"

if not pc.has_index(name=index_name):
    pc.create_index(
        name=index_name,
        metric="cosine",
        dimension=3072,
        spec=ServerlessSpec(
            cloud=CloudProvider.AWS,
            region=AwsRegion.US_EAST_1
        )
    )

index = pc.Index(name=index_name)

Nosso índice agora está pronto, mas está vazio. Ele é um índice vetorial, então precisa de vetores. Como mencionado, para criar esses vetores de embedding usaremos o modelo gemini-embedding-001 do Gemini. Podemos acessá-lo pelo LangChain assim:

In [115]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

Usando esse modelo, podemos gerar embeddings assim:

In [83]:
texts = [
    'Esse é a primeira parcela do texto',
    'agora essa é a segunda parcela do texto'
]

res = embed_model.embed_documents(texts)
len(res), len(res[0])

(2, 3072)

Agora dividimos nossos documentos em partes menores, geramos embeddings para cada parte e enviamos tudo para o índice.

In [89]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [96]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 183 sub-documents.


Cada é convertido em vetor com o modelo de embeddings e depois inserido no índice vetorial para que possa ser recuperado pelo nosso chatbot posteriormente.

In [ ]:
for d in all_splits:
    d.metadata["language"] = "pt"
    _id = f"{d.metadata.get('doi', 'no-doi')}-{d.metadata.get('start_index')}"
    text = d.page_content
    embed = embed_model.embed_documents([text])[0]

    metadata = {
        "text": text,
        **d.metadata
    }

    index.upsert(vectors=[(_id, embed, metadata)])


odemos verificar que o índice vetorial foi populado usando describe_index_stats, exatamente como antes.

In [106]:
index.describe_index_stats()

{'dimension': 3072,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 101}},
 'total_vector_count': 101,
 'vector_type': 'dense'}

#### Retrieval Augmented Generation (RAG)

Criamos uma base de conhecimento completa. Agora precisamos conectá-la ao nosso chatbot. Para isso, voltamos ao LangChain e reutilizamos o template de prompt que usamos antes.

Para usar o LangChain aqui, precisamos carregar a abstração do índice vetorial (vectorstore).
Inicializamos esse objeto passando o nosso índice vetorial já criado.

In [107]:
from langchain_pinecone import PineconeVectorStore

text_field = "text"

vectorstore = PineconeVectorStore(
    index=index,
    embedding=embed_model,
    text_key=text_field
)

Com esse vectorstore, já conseguimos consultar o índice e verificar se existe alguma informação relevante para a nossa pergunta sobre a Lei 13.756/2018.

In [150]:
query = "De acordo com a Lei 13.756/2018, quais entidades esportivas podem receber recursos das loterias e que tipo de prestação de contas elas devem apresentar para garantir a integridade no uso desses recursos?"

vectorstore.similarity_search(query, k=3)

[Document(id='no-doi-4', metadata={'language': 'pt', 'source': 'https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13756.htm', 'start_index': 4.0, 'title': '\n   L13756\n  '}, page_content='L13756\n  \n           Presidência da República\n          \n           Secretaria-Geral\n          \n           Subchefia para Assuntos Jurídicos\n          \n       LEI Nº 13.756, DE 12 DE DEZEMBRO DE 2018.\n      \n          Vigência\n         \n          Regulamento\n         \n          Conversão da Medida Provisória nº 846, de 2018\n         \nTexto compilado\n(Vide Medida \n\t\tProvisória nº 1.303, de 2025)\xa0\xa0\n\t\tProdução de efeitos\n\xa0\xa0\xa0\r\n\t\r\n\tVigência encerrada\n         Dispõe sobre o Fundo Nacional de Segurança Pública (FNSP), sobre a destinação do produto da arrecadação das loterias e sobre a promoção comercial e a modalidade lotérica denominada apostas de quota fixa; altera as Leis n\n         \n          º\n         \n         8.212, de 24 de julho de 199

A consulta retorna bastante texto e não fica claro o que é realmente relevante Felizmente, o LLM consegue interpretar esse conteúdo muito mais rápido do que nós.
Tudo o que precisamos é conectar a saída do nosso vectorstore ao nosso chatbot.
Para isso, usamos a mesma lógica que aplicamos anteriormente.

In [151]:
def augment_prompt(query: str):
    results = vectorstore.similarity_search(query, k=5)
    source_knowledge = "\n".join([x.page_content for x in results])
    augmented_prompt = f"""Usando os contextos abaixo, responda a pergunta.

    Contextos:
    {source_knowledge}

    Pergunta: {query}"""
    return augmented_prompt

Com isso, produzimos um prompt aumentado, combinando a pergunta original do usuário com o conteúdo recuperado do nosso vectorstore. Isso permite que o LLM responda usando tanto seu conhecimento paramétrico quanto as informações da nossa base de conhecimento.

In [152]:
print(augment_prompt(query))

Usando os contextos abaixo, responda a pergunta.

    Contextos:
    L13756
  
           Presidência da República
          
           Secretaria-Geral
          
           Subchefia para Assuntos Jurídicos
          
       LEI Nº 13.756, DE 12 DE DEZEMBRO DE 2018.
      
          Vigência
         
          Regulamento
         
          Conversão da Medida Provisória nº 846, de 2018
         
Texto compilado
(Vide Medida 
		Provisória nº 1.303, de 2025)  
		Produção de efeitos
   
	
	Vigência encerrada
         Dispõe sobre o Fundo Nacional de Segurança Pública (FNSP), sobre a destinação do produto da arrecadação das loterias e sobre a promoção comercial e a modalidade lotérica denominada apostas de quota fixa; altera as Leis n
         
          º
         
         8.212, de 24 de julho de 1991, 9.615, de 24 março de 1998, 10.891, de 9 de julho de 2004, 11.473, de 10 de maio de 2007, e 13.675, de 11 de junho de 2018; e revoga dispositivos das Leis n
         
          º
de

Ainda há muito texto no prompt aumentado, então vamos repassá-lo para o nosso modelo de chat e ver como ele se sai ao responder usando essas informações.

In [153]:
prompt = HumanMessage(
    content=augment_prompt(query)
)

messages.append(prompt)

res = chat.invoke(messages)

print(res.content)

De acordo com a Lei nº 13.756, de 12 de dezembro de 2018, as seguintes entidades esportivas podem receber recursos das loterias:

**Entidades Esportivas Beneficiárias (Art. 22):**

1.  **COB** (Comitê Olímpico do Brasil)
2.  **CPB** (Comitê Paralímpico Brasileiro)
3.  **CBC** (Comitê Brasileiro de Clubes)
4.  **CBDE** (Confederação Brasileira do Desporto Escolar)
5.  **CBDU** (Confederação Brasileira do Desporto Universitário)
6.  **Fenaclubes** (Federação Nacional dos Clubes)
7.  **Secretarias estaduais de esporte ou órgãos equivalentes**
8.  **Organizações de prática esportiva da modalidade futebol** (em contrapartida ao uso de suas denominações, suas marcas, seus emblemas, seus hinos ou seus símbolos para divulgação e execução do concurso de prognóstico específico e da Lotex).
9.  **Entidades do Sistema Nacional do Esporte** e **atletas brasileiros ou vinculados a organizações de prática desportiva sediada no País** (em contrapartida ao uso de suas denominações, seus apelidos despor

## Conclusão

Durante a etapa inicial de validação do chatbot com RAG, utilizamos um exemplo mínimo envolvendo apenas um documento para verificar o funcionamento básico do pipeline. Porém, a chave de API do Google Gemini disponibilizada por um aluno não foi capaz de sustentar nem mesmo esse teste simples.

Nas primeiras operações de geração de embeddings e nas chamadas iniciais ao modelo, toda a quota disponível foi consumida rapidamente, interrompendo o processo antes da conclusão. O uso do Google Colab também contribuiu para a inviabilidade do teste. Embora seja um ambiente prático para experimentação rápida, ele não oferece estabilidade de sessão, perde dependências após reinicializações e não mantém persistência adequada para a continuidade de pipelines que exigem várias etapas sequenciais.

Dessa forma, fica evidente que chaves de API pessoais ou estudantis, com quotas limitadas, não atendem às necessidades mínimas de testes funcionais de um sistema baseado em RAG. Da mesma maneira, o Google Colab não fornece condições adequadas para um ambiente básico de validação e está ainda mais distante dos requisitos necessários para um ambiente de homologação, que demanda estabilidade, continuidade operacional e disponibilidade garantida de recursos.

Apesar disso testes básicos são sim eficientes no ambiente do Google Colab, mas de fato não é a solução ideal para o problema apresentado pelo projeto da disciplina.